### Collect data

In [5]:
import pandas as pd
datacol = pd.read_csv('./data/benchmark_24-replicas.csv')
datacol.head()

,adapter,directory,path,replica_no,replicas,name,p,n,description,p_informative
0,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,1,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
1,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,2,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
2,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,3,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
3,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,4,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"
4,rebate,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,./data/rebate/CA_Datasets_Loc_2_Qnt_2_Pop_1000...,5,24,Continuous Features,20,1600,"Continuous Features [p=20,n=1600,p_=2,h=0.4,ed...","18,19"


In [6]:
from tqdm import tqdm

rankdata = pd.DataFrame()
for _, dataset in tqdm(datacol.iterrows(), total=len(datacol)):
    rankpath = os.path.join('../../results/11714894/ranking', dataset.path + '.csv')
    if os.path.exists(rankpath):
        data = pd.read_csv(rankpath)
        # add more specific info to the ranking rows; e.g. replica_no, etc.
        rich_data = data.assign(**dataset)
        rankdata = rankdata.append(rich_data)
rankdata.to_csv('./data/rankdata.csv', index=False)
print(rankdata.shape)

100%|██████████| 1828/1828 [01:10<00:00, 26.06it/s]
(182734, 14)


In [7]:
import numpy as np
from sklearn.utils import Bunch

statdata = pd.DataFrame() # data, statistically post-processed
group_keys = ['description', 'ranking_method', 'replica_no']
for keys, rankset in tqdm(rankdata.groupby(group_keys)):
    description, ranking_method, replica_no = keys
    p_informatives = np.unique(rankset['p_informative'].values) # takes long for NaN's..
    if pd.isna(p_informatives).any(): # no `p_informative` set
        continue

    p_informative = [int(x) for x in p_informatives[0].split(',')]
    for feature_rank, ranks in rankset.groupby('feature_rank'):
        subset_rows = rankset[rankset['feature_rank'] <= feature_rank]
        subset = subset_rows['feature_index'].values

        stat = Bunch()
        stat.n_positives = len(subset)
        stat.n_true_positives = np.sum(np.isin(subset, p_informative))
        stat.n_false_positives = stat.n_positives - stat.n_true_positives
        stat.n_false_negatives = len(p_informative) - stat.n_true_positives
        stat.recall = stat.n_true_positives / len(p_informative)
        stat.precision = stat.n_true_positives / stat.n_positives
        statdata = statdata.append(ranks.assign(**stat))
statdata.to_csv('./data/statdata.csv', index=False)
statdata.shape

100%|██████████| 9150/9150 [1:26:08<00:00,  1.77it/s]


(157038, 20)